# Full partition experiment — all 21 F/T/S configurations

Runs every valid monotonic F/T/S partition of ResNet50 on iWildCam.  
Each backbone block is independently assigned **F** (frozen pretrained),
**T** (fine-tuned pretrained), or **S** (scratch); the head is always **S**.

**Valid configurations**: F^a T^b S^c over (stem, stage1, stage2, stage3, stage4),  
a + b + c = 5 → **21 configs** (head always S).

**Metrics collected**:
- ID / OOD accuracy and balanced accuracy (M1–M2)
- ID / OOD macro-F1 (M3–M4)
- Training FLOPs per sample and total (M5, estimated)
- Inference FLOPs (M6, constant for all configs)
- Trainable parameter count, wall-clock time
- `frozen_depth` (a), `scratch_start` (a+b) — structural axes for analysis

In [ ]:
import sys
from pathlib import Path
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, str(Path.cwd().parent))

from config   import CONFIG
from models   import build_model, config_label, enumerate_configs
from evaluate import Evaluator
from budget   import EpochBudgetTracker
from train    import train
from utils    import set_seed, estimate_flops

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS = Path('../results')
RESULTS.mkdir(exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
BASE_CFG = {
    **CONFIG,
    'dataset_mode':         'mini',
    'backbone':             'resnet50',
    'epochs':               10,
    'batch_size':           32,
    'num_workers':          4,
    'lr':                   1e-4,
    'lr_head':              1e-3,
    'weight_decay':         1e-4,
    'scheduler':            'cosine',
    'use_weighted_sampler': False,
    'seed':                 42,
}

ALL_CONFIGS = enumerate_configs()  # list of 21 tuples
LABELS = [config_label(p) for p in ALL_CONFIGS]

print(f'{len(ALL_CONFIGS)} configurations to run × {BASE_CFG["epochs"]} epochs each\n')
print('Label         Partition')
for label, partition in zip(LABELS, ALL_CONFIGS):
    print(f'  {label:10s}  {partition}')

In [ ]:
all_logs   = {}   # label → eval_log  (mid-training snapshots, one entry per epoch)
all_final  = {}   # label → final_results
all_params = {}   # label → (n_trainable, n_total)
all_flops  = {}   # label → flops dict

for partition, label in zip(ALL_CONFIGS, LABELS):
    print(f'\n{"-"*54}')
    print(f'  {label}  {partition}')
    print(f'{"-"*54}')

    set_seed(BASE_CFG['seed'])
    cfg = {**BASE_CFG, 'partition': partition}

    model, n_train, n_total = build_model(partition, cfg)
    all_params[label] = (n_train, n_total)
    print(f'  Trainable: {n_train:,} / {n_total:,}  ({100*n_train/n_total:.1f}%)')

    flops = estimate_flops(model, n_trainable=n_train, n_total=n_total)
    all_flops[label] = flops

    evaluator = Evaluator(cfg, DEVICE)
    tracker   = EpochBudgetTracker(eval_every_n=1, max_epochs=cfg['epochs'])

    log, final = train(
        model, cfg, evaluator, tracker,
        eval_splits=['id_val', 'ood_val'],
        final_splits=['id_val', 'ood_val', 'id_test', 'ood_test'],
        checkpoint_dir=None,
    )
    all_logs[label]  = log
    all_final[label] = final

print('\nAll 21 configurations done.')

In [ ]:
# ── Per-config results CSV ────────────────────────────────────────────────────
N_TRAIN   = sum(1 for _ in open(BASE_CFG.get('data_dir','') + '/iwildcam_mini/metadata.csv')) - 1
N_EPOCHS  = BASE_CFG['epochs']

rows = []
for label, partition in zip(LABELS, ALL_CONFIGS):
    final        = all_final[label]
    n_train, n_t = all_params[label]
    flops        = all_flops[label]

    a  = partition[:5].count('F')   # frozen_depth
    b  = partition[:5].count('T')   # tuned_depth
    c  = partition[:5].count('S')   # scratch_depth (backbone)

    def g(split, key):
        return final.get(split, {}).get(key, float('nan'))

    id_bal  = g('id_val',  'balanced_acc')
    ood_bal = g('ood_val', 'balanced_acc')

    total_gflops = (flops['training_flops_per_sample'] * N_TRAIN * N_EPOCHS) / 1e9

    rows.append({
        'label':                    label,
        'partition':                str(partition),
        'frozen_depth':             a,
        'tuned_depth':              b,
        'scratch_depth':            c,
        'scratch_start':            a + b,
        'trainable_params':         n_train,
        'trainable_pct':            round(100 * n_train / n_t, 1),
        'inference_flops_G':        round(flops['inference_flops'] / 1e9, 2),
        'training_flops_G':         round(flops['training_flops_per_sample'] / 1e9, 2),
        'total_train_GFLOPs':       round(total_gflops),
        'elapsed_s':                all_logs[label][-1]['elapsed_s'] if all_logs[label] else float('nan'),
        'id_val_acc':               g('id_val',  'acc'),
        'id_val_bal_acc':           id_bal,
        'id_val_f1':                g('id_val',  'f1'),
        'ood_val_acc':              g('ood_val', 'acc'),
        'ood_val_bal_acc':          ood_bal,
        'ood_val_f1':               g('ood_val', 'f1'),
        'id_ood_gap':               round(id_bal - ood_bal, 4) if not (id_bal != id_bal) else float('nan'),
        'id_test_acc':              g('id_test',  'acc'),
        'id_test_bal_acc':          g('id_test',  'balanced_acc'),
        'id_test_f1':               g('id_test',  'f1'),
        'ood_test_acc':             g('ood_test', 'acc'),
        'ood_test_bal_acc':         g('ood_test', 'balanced_acc'),
        'ood_test_f1':              g('ood_test', 'f1'),
    })

df = pd.DataFrame(rows).set_index('label')
df.to_csv(RESULTS / 'spectrum_experiment.csv')

print('\n── Results (sorted by OOD val balanced acc) ─────────────────────────')
show = ['frozen_depth','scratch_start','trainable_pct',
        'id_val_bal_acc','ood_val_bal_acc','id_ood_gap','elapsed_s','total_train_GFLOPs']
print(df[show].sort_values('ood_val_bal_acc', ascending=False).to_string())
print(f'\nSaved → {RESULTS / "spectrum_experiment.csv"}')

In [ ]:
# ── Eval-log curves CSV (needed by 05_results_analysis.ipynb) ─────────────────
# Long-format: one row per (label × epoch checkpoint)
curve_rows = []
for label in LABELS:
    for entry in all_logs[label]:
        curve_rows.append({
            'label':          label,
            'epoch':          entry.get('epoch'),
            'step':           entry.get('step'),
            'elapsed_s':      entry.get('elapsed_s'),
            'train_loss':     entry.get('train_loss'),
            'train_acc':      entry.get('train_acc'),
            'id_val_acc':     entry.get('id_val_acc'),
            'id_val_bal_acc': entry.get('id_val_balanced_acc'),
            'id_val_f1':      entry.get('id_val_f1'),
            'ood_val_acc':    entry.get('ood_val_acc'),
            'ood_val_bal_acc':entry.get('ood_val_balanced_acc'),
            'ood_val_f1':     entry.get('ood_val_f1'),
        })

curves_df = pd.DataFrame(curve_rows)
curves_df.to_csv(RESULTS / 'eval_log_all.csv', index=False)
print(f'Eval log saved → {RESULTS / "eval_log_all.csv"}')
print(f'Shape: {curves_df.shape}  ({len(LABELS)} configs × {BASE_CFG["epochs"]} epochs)')
print(curves_df.head(6).to_string())

In [ ]:
# ── Quick plots (raw, unpolished — see 05_results_analysis for final plots) ───
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = plt.cm.RdYlBu_r(df['frozen_depth'] / 5)

axes[0].scatter(df['id_val_bal_acc'], df['ood_val_bal_acc'], c=colors, s=110, zorder=3)
for label in LABELS:
    axes[0].annotate(label, (df.loc[label,'id_val_bal_acc'], df.loc[label,'ood_val_bal_acc']),
                     textcoords='offset points', xytext=(5,3), fontsize=6.5)
lo = min(df['id_val_bal_acc'].min(), df['ood_val_bal_acc'].min()) - 0.02
hi = max(df['id_val_bal_acc'].max(), df['ood_val_bal_acc'].max()) + 0.04
axes[0].plot([lo,hi],[lo,hi],'k--',lw=0.8,alpha=0.4)
axes[0].set_xlim(lo,hi); axes[0].set_ylim(lo,hi)
axes[0].set_xlabel('ID val balanced acc'); axes[0].set_ylabel('OOD val balanced acc')
axes[0].set_title('ID vs OOD  (color = frozen_depth)')
axes[0].grid(True, alpha=0.25)

df_s = df.sort_values('ood_val_bal_acc', ascending=False)
gap  = df_s['id_ood_gap']
axes[1].bar(range(len(gap)), gap.values,
            color=plt.cm.RdYlBu_r(df_s['frozen_depth'] / 5), edgecolor='white')
axes[1].set_xticks(range(len(gap)))
axes[1].set_xticklabels(gap.index, rotation=45, ha='right', fontsize=7)
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_ylabel('ID bal acc − OOD bal acc')
axes[1].set_title('ID–OOD gap (sorted by OOD acc)')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'spectrum_quick_plots.png', bbox_inches='tight')
plt.show()